# 01 — Clean DIABIMMUNE 16S data

Keep bacterial genera, retain subjects with at least 3 16S samples, preserve the full metadata table, and align metadata and counts once at the end.


In [1]:
COUNTS_FILE <- "../data/DIABIMMUNE/diabimmune_data_16s.csv"
METADATA_FILE <- "../data/DIABIMMUNE/diabimmune_metadata.csv"
MIN_SAMPLES <- 3

root <- if (dir.exists("data")) "." else ".."

counts <- read.csv(COUNTS_FILE, check.names = FALSE)
metadata <- read.csv(METADATA_FILE, check.names = FALSE)

metadata$sample_id <- as.character(metadata$SampleID)
metadata$subject_id <- as.character(metadata$subjectID)
metadata$age <- as.numeric(metadata$age_at_collection)
metadata$country <- toupper(as.character(metadata$country))

sample_ids <- as.character(counts[[1]])
counts <- counts[-1]
counts[] <- lapply(counts, as.numeric)

# filter for genus-level bacteria
names0 <- colnames(counts)
keep <- grepl("\\|g__[^|]*$", names0) & !startsWith(names0, "k__Archaea")
counts <- counts[, keep, drop = FALSE]
names0 <- colnames(counts)

# serialize unknown genera
unknown <- grepl("\\|g__$", names0)
names1 <- names0
names1[unknown] <- paste0(
  sub("\\|g__$", "", names0[unknown]),
  "|g__Unknown_", sprintf("%03d", seq_len(sum(unknown)))
)
colnames(counts) <- names1

# function to sort taxa
rank <- function(x, prefix) {
  hit <- strsplit(x, "\\|")[[1]]
  hit <- hit[startsWith(hit, prefix)]
  if (length(hit)) hit[1] else NA_character_
}

# sort taxa for greengenes taxa tree
taxonomy <- data.frame(
  feature = names1,
  kingdom = sapply(names0, rank, "k__"),
  phylum = sapply(names0, rank, "p__"),
  class = sapply(names0, rank, "c__"),
  order = sapply(names0, rank, "o__"),
  family = sapply(names0, rank, "f__"),
  genus = sapply(names1, rank, "g__")
)

In [2]:
# keep metadata with 16S data
metadata <- metadata[metadata$sample_id %in% sample_ids, ]

# find and remove subjects with fewer than 3 samples
visits <- table(metadata$subject_id)
dropped_subjects <- names(visits[visits < MIN_SAMPLES])
metadata <- metadata[!metadata$subject_id %in% dropped_subjects, ]

# sort metadata and align counts once
o <- order(metadata$subject_id, metadata$age)
metadata <- metadata[o, ]
counts <- counts[match(metadata$sample_id, sample_ids), , drop = FALSE]

# save processed data
output <- file.path(root, "data", "processed_16s", format(Sys.time(), "%Y%m%d_%H%M%S"))
dir.create(output, recursive = TRUE)

write.csv(
  data.frame(sample_id = metadata$sample_id, counts, check.names = FALSE),
  file.path(output, "counts.csv"),
  row.names = FALSE
)
write.csv(metadata, file.path(output, "metadata.csv"), row.names = FALSE)
write.csv(taxonomy, file.path(output, "taxonomy.csv"), row.names = FALSE)

cat("Dropped", length(dropped_subjects), "subjects with fewer than", MIN_SAMPLES, "samples:\n")
print(dropped_subjects)

cat(
  nrow(metadata), "samples |",
  length(unique(metadata$subject_id)), "subjects |",
  ncol(counts), "genera |",
  ncol(metadata), "metadata columns\n"
)

cat("Saved:", output, "\n")


Dropped 12 subjects with fewer than 3 samples:
 [1] "E002410" "E002825" "E024213" "E025830" "P002644" "P003825" "P004447"
 [8] "P007441" "P015656" "T005093" "T014825" "T014976"
1564 samples | 209 subjects | 143 genera | 63 metadata columns
Saved: ../data/processed_16s/20260811_212353 
